In [1]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from redbox.chains.components import get_chat_llm
from redbox.models.settings import ChatLLMBackend
from redbox.graph.nodes.tools import build_govuk_search_tool, build_search_documents_tool, build_search_wikipedia_tool
from langchain_core.tools import StructuredTool
from redbox.models.settings import Settings
from redbox.models.file import ChunkResolution


INFO:root:the parsed url is ParseResult(scheme='http', netloc='admin:Opensearch2024^@localhost:9200', path='', params='', query='', fragment='')


In [2]:
# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react")

/Users/noraerrouhly/Documents/redbox/django_app/.venv/lib/python3.12/site-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [27]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [28]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [3]:
#llm = get_chat_llm(state.request.ai_settings.chat_backend, tools=tools)

chat_backend = ChatLLMBackend(name="anthropic.claude-3-sonnet-20240229-v1:0", provider="bedrock")

 # Tools

_env = Settings()

search_documents = build_search_documents_tool(
            es_client=_env.elasticsearch_client(),
            index_name=_env.elastic_chunk_alias,
            embedding_model=_env.embedding_backend,
            embedding_field_name=_env.embedding_document_field_name, #not used
            chunk_resolution=ChunkResolution.normal,
        )
search_wikipedia = build_search_wikipedia_tool()
search_govuk = build_govuk_search_tool()
        
agent_tool_names = ["_search_documents", "_search_wikipedia", "_search_govuk"]

tools: dict[str, StructuredTool] = {
            "_search_documents": search_documents,
            "_search_govuk": search_govuk,
            "_search_wikipedia": search_wikipedia,
        }
agent_tools: list[StructuredTool] = [tools[tool_name] for tool_name in agent_tool_names]

agent_tools: list[StructuredTool] = [search_documents]



INFO:root:Testing OpenSearch is definitely being used
INFO:opensearch:HEAD http://localhost:9200/_alias/redbox-data-integration-chunk-current [status:200 request:0.009s]
INFO:opensearch:HEAD http://localhost:9200/redbox-data-integration-chat-mesage-log [status:200 request:0.020s]


TypeError: tool() got an unexpected keyword argument 'description'

In [6]:
search_documents

StructuredTool(name='_search_documents', description="Search for documents uploaded by the user based on a query string.\n\nThis function performs a search over the user's uploaded documents\nand returns snippets from the documents ordered by relevance and\ngrouped by document.\n\nArgs:\n    query (str): The search query string used to match documents.\n        This could be a keyword, phrase, question, or text from\n        the documents.\n\nReturns:\n    dict[str, Any]: A collection of document objects that match the query.", args_schema=<class 'langchain_core.utils.pydantic._search_documents'>, response_format='content_and_artifact', func=<function build_search_documents_tool.<locals>._search_documents at 0x30bb52520>)

In [7]:
agent_tools

[StructuredTool(name='_search_documents', description="Search for documents uploaded by the user based on a query string.\n\nThis function performs a search over the user's uploaded documents\nand returns snippets from the documents ordered by relevance and\ngrouped by document.\n\nArgs:\n    query (str): The search query string used to match documents.\n        This could be a keyword, phrase, question, or text from\n        the documents.\n\nReturns:\n    dict[str, Any]: A collection of document objects that match the query.", args_schema=<class 'langchain_core.utils.pydantic._search_documents'>, response_format='content_and_artifact', func=<function build_search_documents_tool.<locals>._search_documents at 0x30bb52520>)]

In [8]:
llm = get_chat_llm(chat_backend, tools=agent_tools)
# Construct the ReAct agent
agent = create_react_agent(llm, tools, prompt)

AttributeError: 'str' object has no attribute 'name'

In [ ]:
# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({"input": "what is LangChain?"})